# Clasificador de Basura desde Cero (Orgánico vs Reciclable)
Red Neuronal Convolucional (CNN) entrenada **desde cero (From Scratch)**.
- **Entrada:** Imágenes RGB (150x150x3)
- **Capas intermedias:** cp1, cp2, cp3 (bloques convolucionales) y cp4 (densa)
- **Salida:** Probabilidad binaria (0 = Orgánico, 1 = Reciclable)

In [ ]:
# 1. Instalar dependencias y verificar GPU
!pip install -q kagglehub tensorflow matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs disponibles:", gpus if gpus else "Ninguna (usando CPU, activá GPU T4 en Entorno de ejecución)")

In [ ]:
# 2. Descargar dataset desde Kaggle
import kagglehub

path = kagglehub.dataset_download("techsash/waste-classification-data")
print("Dataset descargado en:", path)

# Localizar carpetas TRAIN y TEST automáticamente
train_dir, test_dir = None, None
for root, dirs, files in os.walk(path):
    upper_dirs = [d.upper() for d in dirs]
    if "TRAIN" in upper_dirs and "TEST" in upper_dirs:
        for d in dirs:
            if d.upper() == "TRAIN":
                train_dir = os.path.join(root, d)
            elif d.upper() == "TEST":
                test_dir = os.path.join(root, d)
        break

print("Train dir:", train_dir)
print("Test dir: ", test_dir)

In [ ]:
# 3. Cargar imágenes y preparar datasets (batches de 32 a 150x150)
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Clases detectadas:", class_names)  # 0: Orgánico (O), 1: Reciclable (R)

# Optimización en memoria
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# 4. Arquitectura de la Red: capa_entrada -> cp1 -> cp2 -> cp3 -> cp4 -> capa_salida

# Capa de Entrada (150x150x3) y normalización a [0, 1]
capa_entrada = layers.Input(shape=(150, 150, 3), name="capa_entrada")
norm = layers.Rescaling(1./255, name="normalizacion")(capa_entrada)

# Capa 1 (cp1): Detecta bordes y colores iniciales
cp1 = layers.Conv2D(32, (3, 3), activation="relu", name="cp1_conv")(norm)
cp1 = layers.MaxPooling2D(2, 2, name="cp1_pool")(cp1)

# Capa 2 (cp2): Detecta texturas, esquinas y curvaturas
cp2 = layers.Conv2D(64, (3, 3), activation="relu", name="cp2_conv")(cp1)
cp2 = layers.MaxPooling2D(2, 2, name="cp2_pool")(cp2)

# Capa 3 (cp3): Detecta partes estructurales de botellas, cáscaras, etc.
cp3 = layers.Conv2D(128, (3, 3), activation="relu", name="cp3_conv")(cp2)
cp3 = layers.MaxPooling2D(2, 2, name="cp3_pool")(cp3)

# Aplanado a vector unidimensional
flat = layers.Flatten(name="aplanado")(cp3)

# Capa 4 (cp4): Capa densa de razonamiento y regularización
cp4 = layers.Dense(128, activation="relu", name="cp4_densa")(flat)
cp4 = layers.Dropout(0.5, name="cp4_dropout")(cp4)

# Capa de Salida: probabilidad binaria (0 = Orgánico, 1 = Reciclable)
capa_salida = layers.Dense(1, activation="sigmoid", name="capa_salida")(cp4)

# Ensamble del modelo
model = models.Model(inputs=capa_entrada, outputs=capa_salida, name="ClasificadorBasura_DesdeCero")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# 5. Entrenar la red neuronal (10 épocas)
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

In [ ]:
# 6. Curvas de aprendizaje y guardar modelo
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Entrenamiento")
plt.plot(epochs_range, val_acc, label="Validación")
plt.title("Precisión (Accuracy)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Entrenamiento")
plt.plot(epochs_range, val_loss, label="Validación")
plt.title("Pérdida (Loss)")
plt.legend()
plt.show()

# Guardar pesos y arquitectura
model.save("waste_classifier_scratch.keras")
print("Modelo guardado en waste_classifier_scratch.keras")

In [ ]:
# 7. Función de inferencia con cualquier imagen nueva
def probar_imagen(ruta_imagen):
    img = tf.keras.utils.load_img(ruta_imagen, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    batch = np.expand_dims(arr, axis=0)
    
    prob = float(model.predict(batch)[0][0])
    etiqueta = "Reciclable (R)" if prob >= 0.5 else "Orgánico (O)"
    confianza = prob if prob >= 0.5 else (1.0 - prob)
    
    print(f"Predicción: {etiqueta} ({confianza*100:.2f}% de confianza)")